In [33]:
import os
import rasterio
import time
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from rasterio.warp import calculate_default_transform, reproject, Resampling
import shutil

from concurrent.futures import ThreadPoolExecutor

import time
from tqdm import tqdm

# utils

import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.merge import merge
import geopandas as gpd
from shapely.geometry import box,Polygon
import glob
import warnings
from tqdm import tqdm
import pathlib
import os
import shutil
import matplotlib.pyplot as plt


The script processes the downloaded images to prepare them for predictions, but it doesn’t convert BGR to RGB and doesn't clip them with "clean" geometry (overlap issue). The results from each stage will be saved in a new folder. Once the script is stable, all intermediate folders can be removed so that only the final folder remains.  


step 1: change the suffix to .tif (RAMP request)  
step 2: check projection , reproject to 4326 only if projection is different  
step 3: remove NIR band if needed  


In [34]:
# PATHS - this will also be the order the images to be saved
path_source = "/home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch2/3bit"
path_reproject = "/home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch2/ready_for_clean"
path_remove_nir = "/home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch2/remove_nir"
path_bgr_rgb = "/home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch2/ready_for_clean"
path_to_clean = "/home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch1/clean"
# REMOVE_NIR = False
# CLEAN = True
# BGR_TO_RGB = False
DEST_CRS = "EPSG:4326"
polygon_path = "/home/jovyan/work/imagery/afghanistan/south_afg/hilmand_5/clean_hilmand_5.gpkg"

In [35]:
# gdf=gpd.read_file("/home/jovyan/work/imagery/afghanistan/south_afg/NESH/nesh_correct_path2.gpkg")

In [36]:
# gdf.columns

In [37]:
RUN_REPROJECT   = True
RUN_REMOVE_NIR  = False
RUN_BGR_TO_RGB  = False
RUN_CLEAN       = False

In [38]:
def rename_to_tif(
    directory:str , #path to directory with .TIF or anything that is not .tif located
    ):
    
    count = 0 
    
    count1 = 0
    
    for filename in os.listdir(directory):
        
        old_path = os.path.join(directory, filename)
        
        # skip if it's already .tif
        if filename.endswith(".tif"):
            count = count + 1
            # print(f"file name:{filename} ends with .tif , skipping")
            continue
        
        name, ext = os.path.splitext(filename)
        # print(name,ext)
        
        new_filename = name + ".tif"
        new_path = os.path.join(directory, new_filename)
        # print(new_path)
        
        # print(f"Renaming: {old_path} -> {new_path}")
        os.rename(old_path, new_path)
        
        count1 = count1 +1
        
    print(f"{count} files ends already with .tif , {count1} files were changed")
        
        
    

In [39]:
import os
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import shutil

def reproject_rasters(
        lst_urls: list,        # list of paths to rasters
        dest_crs: str,         # destination CRS (e.g. 'EPSG:4326')
        local_save_path: str,  # path to save output rasters
        ):
    """Reprojects rasters unless they are already in the target CRS."""

    # Define destination CRS
    dstCrs = rasterio.crs.CRS.from_string(dest_crs)

    for url in lst_urls:
        try:
            print("\n==============================")
            print(f"Processing: {url}")

            uniq_name = os.path.basename(url)
            dest_path = os.path.join(local_save_path, uniq_name)

            with rasterio.open(url) as srcRst:
                print(f"Source CRS: {srcRst.crs}")

                # ---- CASE 1: Same CRS → Just copy ----
                if srcRst.crs == dstCrs:
                    print("CRS already matches destination → copying file.")
                    shutil.copyfile(url, dest_path)
                    continue

                # ---- CASE 2: Different CRS → Reproject ----
                print("Reprojecting raster...")

                transform, width, height = calculate_default_transform(
                    srcRst.crs, dstCrs, srcRst.width, srcRst.height, *srcRst.bounds
                )

                kwargs = srcRst.meta.copy()
                kwargs.update({
                    'crs': dstCrs,
                    'transform': transform,
                    'width': width,
                    'height': height
                })

                # Write output raster
                with rasterio.open(dest_path, 'w', **kwargs) as dstRst:
                    for i in range(1, srcRst.count + 1):
                        reproject(
                            source=rasterio.band(srcRst, i),
                            destination=rasterio.band(dstRst, i),
                            src_crs=srcRst.crs,
                            dst_crs=dstCrs,
                            src_transform=srcRst.transform,
                            dst_transform=transform,
                            resampling=Resampling.nearest
                        )

                print(f"Saved reprojected raster → {dest_path}")

        except Exception as e:
            print(f"Error with file {url}: {e}")
            continue


In [40]:

# THIS FUNCTION IS GOOD IF YOU RUN IT ON THE SAME INPUT FOLDE3R AS OUTPUT FOLDER, OTHERWISE IT NEEDS FIX!
def process_file(input_path, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with rasterio.open(input_path) as src:
        if src.count >= 4:
            rgb_data = src.read([1, 2, 3])
            profile = src.profile
            profile.update(count=3)
            with rasterio.open(output_path, "w", **profile) as dst:
                dst.write(rgb_data)
            return f"Saved: {output_path}"
        else:
            return f"Skipped: {input_path}, only {src.count} bands."

def remove_nir_band(input_dir, output_dir, max_workers=4):
    start_time = time.time()
    os.makedirs(output_dir, exist_ok=True)
    files = [f for f in os.listdir(input_dir) if f.lower().endswith(".tif")]

    args = [(os.path.join(input_dir, f), os.path.join(output_dir, f))
            for f in files if not os.path.exists(os.path.join(output_dir, f))]

    if not args:
        print("All files already processed.")
        return

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for res in tqdm(executor.map(lambda p: process_file(*p), args),
                        total=len(args), desc="Processing rasters"):
            results.append(res)

    end_time = time.time()
    for r in results:
        print(r)
    print(f"\nFinished processing {len(args)} files with {max_workers} workers in {end_time - start_time:.2f} seconds.")

In [41]:



# def convert_bgr_to_rgb(filename):
#     in_path = os.path.join(input_folder, filename)
#     out_path = os.path.join(output_folder, filename)

#     with rasterio.open(in_path) as src:
#         img = src.read()  # (C, H, W)
#         profile = src.profile

#     rgb = img[[2, 1, 0], :, :]  # Swap BGR → RGB
#     profile.update(count=3)

#     with rasterio.open(out_path, "w", **profile) as dst:
#         dst.write(rgb)

#     print(f"Saved RGB file: {out_path}")

# # Increase the number of threads for faster processing
# num_workers = 8
# with ThreadPoolExecutor(max_workers=num_workers) as executor:
#     executor.map(convert_bgr_to_rgb, files_to_process)


In [42]:

def run_bgr_rgb(input_path, output_path):
    """
    Convert a raster from BGR → RGB and save to output_path.
    If do_convert=False, simply copies the file without modification.
    """

    with rasterio.open(input_path) as src:
        img = src.read()         # Shape: (C, H, W)
        profile = src.profile

  
        # Swap BGR → RGB (C is [B, G, R] → [R, G, B])
        rgb = img[[2, 1, 0], :, :]

        profile.update(count=3)
    
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(rgb)

    print(f"Saved RGB file: {output_path}")

In [43]:
###NEW OPTIMIZED
# constants

#general strings
GEOM_COL_NAME = 'geometry'
PATH_COL_NAME = 'path_2_2'
TIF_SUFFIX = '.TIF'
GEOJSON_SUFFIX = '.geojson'
CONST_CRS = "4326" 
CRS_3857 = "3857"
# SUFFIX_FOR_TIF_FOLDER = "/*.TIF"
SUFFIX_FOR_TIF_FOLDER = "/*.tif"

LAYER_COL_NAME ='layer'

STR_MINX = 'minx'
STR_MINY = 'miny'
STR_MAXX = 'maxx'
STR_MAXY = 'maxy'

BUILDING_STR = "building"

#|export

def save_raster(
    img:np.ndarray , # ndarray image
    number_of_bands: int , #number of  bands
    transf:tuple ,#transform function of the raster
    paths:str ,#save path raster
    tiff_name:str ,#name of specific file
    crs_img:rasterio.crs.CRS,
    ):  
    
    '''
    Function to save raster in requested path

    '''

    savedir=pathlib.Path(paths)
    savedir.mkdir(parents=True,exist_ok=True)


    with rasterio.open(savedir /tiff_name, 
                                   'w',
                                   driver='COG',
                                   height=img.shape[1],
                                   width=img.shape[2],
                                   count=number_of_bands,
                                   dtype=img.dtype,
                                   crs=crs_img,
                                   nodata=None, # change if data has nodata value
                                   transform=transf) as dst:

            for n in np.arange(0,number_of_bands):
                  dst.write(img[n,:,:],int(n+1))   
                    
    return None

#|export

def get_list_of_rasters(
        raster_storage_path : str , #path where the rasters are stored.
        folder_suffix 
    : str = SUFFIX_FOR_TIF_FOLDER
        ):
        """
        Retrieves the file paths of all images within a directory.

        Args:
        - raster_storage_path (str): The path of the directory containing rasters.

        Returns:
        - List[str]: A list containing the file paths of all images within the directory.

        """

        str_for_glob = raster_storage_path + folder_suffix

        list_of_rasters = glob.glob(rf"{str_for_glob}")

        if len(list_of_rasters)==0:
                warnings.warn("No rasters were found", UserWarning)

        else:
                print(f"Found {len(list_of_rasters)} rasters")
                

        return list_of_rasters
    
#|export

def raster_extent_to_vector(
        raster_path : str , # Path to the raster file for which we want to obtain its extent as a vector representation
        # raster_crs : str = "4326", #crs , deafule is 4326
)-> gpd.GeoDataFrame:
        """
        This function takes a string path to a raster file, extract the extent as vector, and returns its extent as geodataframe.
    
        Parameters:
        raster_path (str): The file path to the raster file (e.g., .tif).
        
        Returns:
        GeoDataFrame: A tuple representing the extent of the raster data as geopandas
        """
        
        src = rasterio.open(raster_path)
        src_src=src.crs
        raster_extent = src.bounds

        polygon_gdf = gpd.GeoDataFrame(geometry = [box(*raster_extent)])
        polygon_gdf=polygon_gdf.set_crs(src_src)
        polygon_gdf = polygon_gdf.to_crs(src_src)
#NEW: RETURN src_crs SO ORIGINAL CRS WILL BE USED LATER
        return polygon_gdf,src_src

#|export

def many_rasters_to_vector_layer(
        path_to_rasters : str , #path to rasters layers
        to_dissolve : bool=False , # when set to true, the polygons of the geodataframe will be dissolved
        # raster_crs : str = "4326", #crs , deafule is 4326 
        ):
        
        raster_files = get_list_of_rasters(path_to_rasters,folder_suffix=SUFFIX_FOR_TIF_FOLDER)
        print(f"found {len(raster_files)} rasters in path")

        print('Extracting rasters extent')
        collect_gdf = []
        
        for file in tqdm(raster_files):
            try:
                #NEW : RETURN raster_crs  
                tmp_gdf,raster_crs = raster_extent_to_vector(raster_path = file)
                tmp_gdf[PATH_COL_NAME] = file
                collect_gdf.append(tmp_gdf)
                # print(len(collect_gdf))
                #tmp_gdf.plot(facecolor="none",edgecolor="red" )
            except Exception as e:
                print(f"Error processing path: {file}")
                print(f"Exception: {e}")
                continue


        df = pd.concat(collect_gdf)
        gdf = gpd.GeoDataFrame(df,geometry=GEOM_COL_NAME)

        
        if to_dissolve:
                gdf = gdf.dissolve()
        #gdf.plot(facecolor="none",edgecolor="red" )
                

        return gdf
    


def get_raster_intersects_with_vector(rasters_path, polygon_path, dest_raster_path):
    
    print("open the tiles geodataframe")
    gdf = gpd.read_file(polygon_path)
    print(f"CRS Of Raster extent shape : {gdf.crs}")
    print(f"total tiles to check: {len(gdf)}")
    
    #load raster paths
    print(f"loading rasters from : {rasters_path}")
    lst_paths = glob.glob(rasters_path + SUFFIX_FOR_TIF_FOLDER)
    
    #check if destination for rasters is exists, if not create it
    if not os.path.exists(dest_raster_path):
        os.makedirs(dest_raster_path)
        
    count, count_copy, count_clip = 0, 0, 0
    
    # open the path that is written for each tile extent vector
    for idx, row in gdf.iterrows():
        
        temp_gdf = gpd.GeoDataFrame([row])
        file_path = row[PATH_COL_NAME]  # get the path from the 'path' column
                                     
        try:

            raster = rasterio.open(file_path)
            print(f"Successfully opened file")
            
            # get the extent of original raster
            one_raster_to_vector, src_src = raster_extent_to_vector(
                raster_path =file_path)
            
            # Plot them
            ax = one_raster_to_vector.plot(color="none", edgecolor="red", linewidth=2, label="Polygon")
            # Plot raster extent on the same axes
            temp_gdf.plot(ax=ax, color="blue", alpha=0.5, label="Raster Extent")
    
            # Add legend and title
            plt.legend()
            # plt.title("Raster Extent vs Polygon Overlap")

            # Show the plot
            plt.show()
            
            #define the destination path and name
            file_name = os.path.basename(file_path)
            file_dest_path = os.path.join(dest_raster_path, file_name)
            print(f"file_name {file_name}")
            print(f"file dest: {file_dest_path}")

                    
            #TODO : ADD CONDITION FOR DATUM THAT IS NOT METRIC 
            #calculate Intersection Over Union
            
            # Union of geometries
            union = temp_gdf.unary_union.union(one_raster_to_vector.unary_union)
            # Intersection of geometries
            intersection = temp_gdf.unary_union.intersection(one_raster_to_vector.unary_union)
            # Areas
            intersection_area = intersection.area
            union_area = union.area
            # IoU
            iou = intersection_area / union_area if union_area != 0 else 0
            print(f"iou value : {iou}")
            
            
            if iou==1:
                print("copying image")

                shutil.copyfile(file_path, file_dest_path)
                count_copy += 1
                print("completed!")
            
            elif 0 < iou < 1:
                print("need to clip")
                masked_raster, transform = mask(raster, 
                                                temp_gdf.geometry, 
                                                invert=False)
                save_raster(img=masked_raster,
                            number_of_bands=3,
                            transf=transform,
                            paths=dest_raster_path,
                            tiff_name=file_name,
                            crs_img=raster.crs
                           )
                print(f"saved to : {dest_raster_path}//{file_name}")
                print("completed!")

                count_clip += 1

                count += 1
                
            else:
                print("iou is weird, skipping")
            


        except Exception as e:
            print(f"Failed to open file at {file_path}: {e}")
    
    print(f"Total files copied: {count_copy}")
    print(f"Total files clipped: {count_clip}")
    print(f"Total files pre-processed: {count_copy+count_clip}")
    
    

In [44]:
def get_missing(src_dir, dst_dir):
    src = set(list_files(src_dir))
    dst = set(list_files(dst_dir)) if os.path.exists(dst_dir) else set()
    return sorted(list(src - dst))

In [45]:

def list_files(path):
    return sorted([
        f for f in os.listdir(path)
        if os.path.isfile(os.path.join(path, f))
    ])


def preprocess_from_a_to_z(
    source_path: str = path_source,
    path_reproject: str = path_reproject,
    path_remove_nir: str = path_remove_nir,
    path_bgr_rgb: str = path_bgr_rgb,
    path_to_clean: str = path_to_clean,
    polygon_path: str = polygon_path,
):

    print("=== PREPROCESS PIPELINE START ===")

    # ----------------------------------
    # PRE-STEP: rename to .tif
    # ----------------------------------
    rename_to_tif(directory=source_path)

    # This variable ALWAYS points to the latest valid output
    current_path = source_path

    # ----------------------------------
    # STEP 1: REPROJECT
    # ----------------------------------
    if RUN_REPROJECT:
        print("=== STEP 1: REPROJECT ===")
        os.makedirs(path_reproject, exist_ok=True)

        missing = get_missing(current_path, path_reproject)
        print(f"Missing {len(missing)} rasters")

        if missing:
            reproject_rasters(
                lst_urls=[os.path.join(current_path, f) for f in missing],
                dest_crs=DEST_CRS,
                local_save_path=path_reproject
            )

        current_path = path_reproject
    else:
        print("=== STEP 1 SKIPPED ===")

    # ----------------------------------
    # STEP 2: REMOVE NIR
    # ----------------------------------
    if RUN_REMOVE_NIR:
        print("=== STEP 2: REMOVE NIR ===")
        os.makedirs(path_remove_nir, exist_ok=True)

        missing = get_missing(current_path, path_remove_nir)
        print(f"Missing {len(missing)} rasters")

        for f in tqdm(missing):
            process_file(
                os.path.join(current_path, f),
                os.path.join(path_remove_nir, f)
            )

        current_path = path_remove_nir
    else:
        print("=== STEP 2 SKIPPED ===")

    # ----------------------------------
    # STEP 3: BGR → RGB
    # ----------------------------------
    if RUN_BGR_TO_RGB:
        print("=== STEP 3: BGR → RGB ===")
        os.makedirs(path_bgr_rgb, exist_ok=True)

        missing = get_missing(current_path, path_bgr_rgb)
        print(f"Missing {len(missing)} rasters")

        for f in tqdm(missing):
            run_bgr_rgb(
                os.path.join(current_path, f),
                os.path.join(path_bgr_rgb, f)
            )

        current_path = path_bgr_rgb
    else:
        print("=== STEP 3 SKIPPED ===")

    # ----------------------------------
    # STEP 4: INTERSECT & CLIP
    # ----------------------------------
    if RUN_CLEAN:
        print("=== STEP 4: INTERSECT & CLIP ===")
        os.makedirs(path_to_clean, exist_ok=True)

        src_files = list_files(current_path)
        dst_files = list_files(path_to_clean)

        missing = sorted(set(src_files) - set(dst_files))
        print(f"Missing {len(missing)} rasters to clean")

        if not missing:
            print("All rasters already processed in Step 4 ✔")
            return

        # IMPORTANT: run intersection ONCE, not per file
        get_raster_intersects_with_vector(
            rasters_path=current_path,
            polygon_path=polygon_path,
            dest_raster_path=path_to_clean
        )

        # Validate
        if len(list_files(path_to_clean)) == len(src_files):
            print("Step 4 completed successfully.")
        else:
            print("⚠ Step 4 incomplete.")
    else:
        print("=== STEP 4 SKIPPED ===")

    print("=== PIPELINE FINISHED ===")


In [46]:
preprocess_from_a_to_z()

=== PREPROCESS PIPELINE START ===
0 files ends already with .tif , 46 files were changed
=== STEP 1: REPROJECT ===
Missing 46 rasters

Processing: /home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch2/3bit/24JUL11060331-S3DS_R01C1-200012059847_01_P001.tif
Source CRS: EPSG:4326
CRS already matches destination → copying file.

Processing: /home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch2/3bit/24JUL11060331-S3DS_R03C2-200012059847_01_P001.tif
Source CRS: EPSG:4326
CRS already matches destination → copying file.

Processing: /home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch2/3bit/24JUL11060331-S3DS_R04C1-200012059847_01_P001.tif
Source CRS: EPSG:4326
CRS already matches destination → copying file.

Processing: /home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch2/3bit/24JUL11060331-S3DS_R04C2-200012059847_01_P001.tif
Source CRS: EPSG:4326
CRS already matches destination → copying file.

Processing: /home/jovyan/work/imagery/afghanistan/south_afg/zabul/bat

In [16]:
import glob

len(glob.glob("/home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch1/ready_for_clean/*"))

89

In [2]:
import geopandas as gpd

gdf = gpd.read_file("/home/jovyan/work/predictions/south_afg/batch13/a/geojson/predicted_buildings.geojson")
gdf.to_file("/home/jovyan/work/predictions/south_afg/batch13/a/geojson/batch13_a.gpkg")

/opt/conda/lib/python3.9/site-packages/geopandas/io/file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


In [3]:
import geopandas as gpd

gdf = gpd.read_file("/home/jovyan/work/predictions/south_afg/batch13/b/geojson/predicted_buildings.geojson")
gdf.to_file("/home/jovyan/work/predictions/south_afg/batch13/b/geojson/batch_113_b.gpkg")

/opt/conda/lib/python3.9/site-packages/geopandas/io/file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


In [5]:
import geopandas as gpd

gdf = gpd.read_file("/home/jovyan/work/predictions/south_afg/batch13/c/geojson/predicted_buildings.geojson")
gdf.to_file("/home/jovyan/work/predictions/south_afg/batch13/c/geojson/batch_113_c.gpkg")

/opt/conda/lib/python3.9/site-packages/geopandas/io/file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,


In [17]:
import glob
import os

folder1 = "/home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch1/clean"
folder2 = "/home/jovyan/work/imagery/afghanistan/south_afg/zabul/batch1/ready_for_clean"

# Get file names (not full paths)
files1 = {os.path.basename(f) for f in glob.glob(os.path.join(folder1, "*"))}
files2 = {os.path.basename(f) for f in glob.glob(os.path.join(folder2, "*"))}

# Files only in folder1
only_in_1 = files1 - files2

# Files only in folder2
only_in_2 = files2 - files1

print("Only in folder1:")
for f in sorted(only_in_1):
    print(f)

print("\nOnly in folder2:")
for f in sorted(only_in_2):
    print(f)

Only in folder1:
24JUL11055958-S3DS_R1C1-200011942938_01_P001.tif.ovr.tmp

Only in folder2:
23AUG01062645-S3DS_R01C3-200011913537_01_P001.tif
23NOV23062913-S3DS_R10C1-200011913551_01_P001.tif
24JUL11055958-S3DS_R1C1-200011942938_01_P001.tif
24JUL11055958-S3DS_R1C2-200011942938_01_P001.tif
